# 🧪 Sesión 13 · Manejo de Sesiones Stateful vs Stateless Tokens

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo recuerda el servidor que ya iniciaste sesión?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/login  {"usuario":"ana","password":"secreta"}
HTTP/1.1 200 OK
Set-Cookie: session_id=...; HttpOnly; Path=/; SameSite=Lax
{"mensaje":"sesión iniciada"}

GET /api/perfil   (con la cookie)
HTTP/1.1 200 OK
{"usuario":"ana"}

POST /api/logout  (con la cookie)
HTTP/1.1 204 No Content

GET /api/perfil   (después del logout)
HTTP/1.1 401 Unauthorized
```

_La cookie viaja sola: el cliente no tiene que copiar el identificador en cada petición._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar una sesión con estado en el servidor.
- Emitir una cookie segura y leerla en peticiones posteriores.
- Diferenciar sesión con estado de un token sin estado.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Login correcto y cookie HttpOnly → HTTP 200
- [ ] El perfil se obtiene con la cookie → HTTP 200
- [ ] El logout cierra la sesión → HTTP 204
- [ ] Después del logout el perfil es 401 → HTTP 401

## 🧩 El reto

Implementa login, perfil y logout usando sesiones en memoria y una cookie HttpOnly.

**Requisitos del contrato**

- El usuario de prueba es ana con contraseña secreta.
- POST /api/login valida usuario y contraseña y responde 200.
- La respuesta de login incluye una cookie session_id con HttpOnly y SameSite=Lax.
- GET /api/perfil devuelve el usuario si la cookie es válida; si no, 401.
- POST /api/logout invalida la sesión y responde 204.
- Después del logout, /api/perfil vuelve a responder 401.

> **Pista conceptual:** El cliente solo guarda un identificador opaco. Todo lo demás (usuario, expiración, permisos) vive en el servidor.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion13.md` con la diferencia entre sesión y token.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:

%%writefile app.py
from fastapi import FastAPI, Cookie

app = FastAPI()

# Sesiones guardadas en memoria
sesiones = {}

@app.post("/api/login")
def login(usuario: str, password: str):

    if usuario == "ana" and password == "secreta":
        session_id = "12345"
        sesiones[session_id] = usuario

        return {"mensaje": "sesión iniciada", "session_id": session_id}

    return {"mensaje": "datos incorrectos"}


@app.get("/api/perfil")
def perfil(session_id: str = None):

    if session_id in sesiones:
        return {"usuario": sesiones[session_id]}

    return {"mensaje": "sesión no encontrada"}


@app.post("/api/logout")
def logout(session_id: str = None):

    if session_id in sesiones:
        del sesiones[session_id]

    return {"mensaje": "sesión cerrada"}



Writing app.py


In [8]:

%%writefile app.py
from fastapi import FastAPI, HTTPException, Response, Cookie
from pydantic import BaseModel
import secrets

app = FastAPI(title="Sistema de Sesiones")

# Almacenamiento de sesiones en memoria
sesiones = {}

# Usuario de prueba
USUARIO = "ana"
PASSWORD = "secreta"


# Modelo para recibir los datos del login
class LoginRequest(BaseModel):
    usuario: str
    password: str


# Ruta para iniciar sesión
@app.post("/api/login")
def login(datos: LoginRequest, response: Response):

    # Validar usuario y contraseña
    if datos.usuario != USUARIO or datos.password != PASSWORD:
        raise HTTPException(
            status_code=401,
            detail="Usuario o contraseña incorrectos"
        )

    # Crear identificador único de sesión
    session_id = secrets.token_urlsafe(32)

    # Guardar la sesión en el servidor
    sesiones[session_id] = datos.usuario

    # Enviar cookie al cliente
    response.set_cookie(
        key="session_id",
        value=session_id,
        httponly=True,
        samesite="lax",
        path="/"
    )

    return {"mensaje": "sesión iniciada"}


# Ruta para consultar el perfil
@app.get("/api/perfil")
def perfil(session_id: str = Cookie(default=None)):

    # Comprobar si la sesión existe
    if not session_id or session_id not in sesiones:
        raise HTTPException(
            status_code=401,
            detail="Sesión no válida"
        )

    # Devolver información del usuario
    return {"usuario": sesiones[session_id]}


# Ruta para cerrar sesión
@app.post("/api/logout")
def logout(
    response: Response,
    session_id: str = Cookie(default=None)
):

    # Eliminar la sesión del servidor
    if session_id:
        sesiones.pop(session_id, None)

    # Eliminar la cookie del cliente
    response.delete_cookie(
        key="session_id",
        path="/",
        httponly=True,
        samesite="lax"
    )

    # Responder con HTTP 204 sin contenido
    response.status_code = 204
    response.body = b""

    return response



Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | 	Estoy trabajando con Python y FastAPI en Google Colab. Utilizo sesiones guardadas en memoria y cookies para controlar el inicio y cierre de sesión. |
| **Rol** | La IA debe actuar como un desarrollador de aplicaciones web con conocimientos de Python, FastAPI y manejo de sesiones. |
| **Especificación** | Necesito crear tres rutas: /api/ login para validar el usuario ana y la contraseña secreta; /api/ perfil para mostrar el usuario cuando exista una sesión válida; y /api/ logout para cerrar la sesión. El inicio debe responder HTTP 200, el perfil HTTP 200, el cierre HTTP 204 y después del cierre el perfil debe responder HTTP 401. También se debe utilizar una cookie llamada session_id con HttpOnly y SameSite=Lax. |
| **Output** | 	Espero recibir el código completo del archivo app.py, organizado y comentado, utilizando FastAPI, modelos de Pydantic, sesiones en memoria y cookies HttpOnly. El código debe cumplir los cuatro criterios de aceptación del verificador. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [9]:
from verificar import probar_http

casos = [{'descripcion': 'Login correcto y cookie HttpOnly',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'headers_contiene': {'set-cookie': ['httponly', 'samesite']}},
 {'descripcion': 'El perfil se obtiene con la cookie',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'estado': 200,
  'json_contiene': {'usuario': 'ana'}},
 {'descripcion': 'El logout cierra la sesión', 'metodo': 'POST', 'ruta': '/api/logout', 'estado': 204},
 {'descripcion': 'Después del logout el perfil es 401', 'metodo': 'GET', 'ruta': '/api/perfil', 'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. Login correcto y cookie HttpOnly
✅ 02. El perfil se obtiene con la cookie
✅ 03. El logout cierra la sesión
✅ 04. Después del logout el perfil es 401
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion13.md` a `auditorias/auditoria_sesion13.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   El resultado que más me costó alcanzar fue el cierre de sesión (logout), porque aunque el servidor eliminaba la sesión, no estaba devolviendo correctamente el código HTTP 204 que solicitaba el verificador. Tuve que revisar la respuesta de la ruta y corregirla para que no devolviera contenido.

   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí utilizar secrets.token_urlsafe(32) para generar un identificador único y aleatorio para cada sesión. También utilicé un diccionario para guardar las sesiones en memoria y poder consultar o eliminar los identificadores cuando fuera necesario.

   - ¿Qué error detectó la verificación que tú no habías previsto?
   La verificación detectó que la ruta de cierre de sesión no estaba devolviendo el código HTTP 204 esperado. Al principio, los otros tres casos funcionaban correctamente, pero el logout presentaba ese problema. Después de corregirlo, volví a ejecutar la verificación para comprobar que los cuatro casos cumplieran con los resultados solicitados.
   
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
